# Spanish Electricity Spot Market — 2025 Volatility Metrics

Calculates three volatility KPIs from `master_dataset.csv` for use in the presentation.

**Data structure (discovered during analysis):**
- Granularity: **daily** — one row per calendar day, no intra-day hourly data.
- Date range: 2020-01-01 → 2025-12-31 (2 192 rows total).
- Spot price column: `Spot_Price_SPEL` — daily average Spanish electricity spot price in **€/MWh**.
- 2025 subset: **365 days** (complete year).

**Methodology (because data is daily, not hourly):**
- *Max Day-to-Day Difference* → largest `|price_today − price_yesterday|` across all consecutive day pairs in 2025.
- *Mean / Max Daily Volatility* → rolling 2-day standard deviation (`window=2`) on the 2025 price series; then take the mean and max of that series. This is the prescribed fallback when intra-day data is unavailable.

In [ ]:
import pandas as pd

# ── 1. Load ──────────────────────────────────────────────────────────────────
df = pd.read_csv(
    "data/processed/master_dataset.csv",
    parse_dates=["Date"]
)

print(f"Full dataset  : {df.shape[0]:,} rows | {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"Spot column   : 'Spot_Price_SPEL'  (daily average, €/MWh)")

In [ ]:
# ── 2. Filter strictly for 2025 ──────────────────────────────────────────────
df2025 = (
    df[df["Date"].dt.year == 2025]
    .sort_values("Date")
    .reset_index(drop=True)
)

print(f"2025 subset   : {len(df2025)} rows | {df2025['Date'].min().date()} → {df2025['Date'].max().date()}")
df2025[["Date", "Spot_Price_SPEL"]].head()

In [ ]:
# ── 3. Calculate volatility metrics ──────────────────────────────────────────

# Metric 1 — Max Day-to-Day Difference
# Absolute change between each day and the previous day; take the maximum.
df2025["day_to_day_diff"] = df2025["Spot_Price_SPEL"].diff().abs()
max_d2d = df2025["day_to_day_diff"].max()

# Metrics 2 & 3 — Rolling 2-day standard deviation
# No hourly data exists, so we approximate intra-period volatility with a
# 2-day rolling window: each value captures the dispersion between that day
# and its immediate predecessor.
df2025["rolling_2d_std"] = df2025["Spot_Price_SPEL"].rolling(window=2).std()

mean_vol = df2025["rolling_2d_std"].mean()  # NaN at row 0 is excluded automatically
max_vol  = df2025["rolling_2d_std"].max()

# Preview the computed columns
df2025[["Date", "Spot_Price_SPEL", "day_to_day_diff", "rolling_2d_std"]].head(10)

In [ ]:
# ── 4. Final results — copy these numbers for the presentation ───────────────
print("=" * 54)
print("   2025 Spanish Electricity Spot Market — Volatility")
print("=" * 54)
print(f"   Max Day-to-Day Difference : {max_d2d:>8.2f}  €/MWh")
print(f"   Mean Daily Volatility     : {mean_vol:>8.2f}  €/MWh")
print(f"   Max Daily Volatility      : {max_vol:>8.2f}  €/MWh")
print("=" * 54)
print()
print(f"  Sample size : {len(df2025)} days (full year 2025)")
print( "  Volatility  : rolling 2-day standard deviation (daily data)")